# Data Pre-processing

In [ ]:
import sklearn as sk
import pandas as pd
import seaborn as sn
import matplotlib.pyplot as plt

## Import Data

In [ ]:
df = pd.read_csv("US_Accidents_March23.csv")
len(df)

## Basic Data Preprocessing
To simplify your project a bit, use only the data instances that contain at most 1 missing value. According to our counts:
There are 3,554,549 data instances with no missing values:
There are 244,702 data instances with exactly one missing value.
Hence, your dataset for the project should contain 3,554,549 + 244,702 = 3,799,251

In [ ]:
df.dropna(thresh=len(df.columns) - 1, inplace=True)
len(df)

In [ ]:
df.drop(columns=['ID', 'Description', 'Country', 'Source', 'Turning_Loop'], inplace = True)
print(df.columns)
basic_df = df.copy(deep=True)

## Advanced Pre-processing
Goals: 
- Encode all features to numeric values
- Use feature selection/elimination to reduce dimensionality for later analysis

In [ ]:
# basic_df is already an independent copy; reuse df to avoid another full-data copy.
advanced_df = df.copy(deep = True)

# Handle datetime -> integer by extracting integer values
advanced_df["Start_Time"] = pd.to_datetime(advanced_df["Start_Time"], format="ISO8601")
advanced_df["End_Time"] = pd.to_datetime(advanced_df["End_Time"], format="ISO8601")

advanced_df["Duration_Minutes"] = (
    advanced_df["End_Time"] - advanced_df["Start_Time"]
).dt.total_seconds() / 60.0
advanced_df["Start_Hour"] = advanced_df["Start_Time"].dt.hour
advanced_df["Start_DayOfWeek"] = advanced_df["Start_Time"].dt.dayofweek  # 0 = Monday, 6 = Sunday
advanced_df["Start_Month"] = advanced_df["Start_Time"].dt.month
advanced_df.drop(columns=["Start_Time", "End_Time"], inplace=True)

In [ ]:
# Drop location cols other than lat/long
advanced_df.drop(
    columns=[
        "City", "County", "Zipcode", "Street", "Airport_Code", "State"
    ],
    inplace=True,
)

In [ ]:
#Convert sunset measures
sunset_fields = ["Sunrise_Sunset", "Civil_Twilight", "Nautical_Twilight", "Astronomical_Twilight"]
for f in sunset_fields:
    advanced_df[f] = advanced_df[f].map({"Day": 1, "Night": 0}) # day = 1, night = 0

In [ ]:
# One-hot encode wind direction and weather condition as dense columns
weather_columns = ["Wind_Direction", "Weather_Condition"]
advanced_df = pd.get_dummies(
    advanced_df,
    columns=weather_columns,
    dummy_na=True,
)

In [ ]:
#Drop timezone, weather
advanced_df.drop(columns = ["Timezone", "Weather_Timestamp"], inplace = True)

In [ ]:
# Fill missing numeric values one column at a time to avoid allocating another full-size frame.
for column in advanced_df.columns:
    missing = advanced_df[column].isna()
    if missing.any():
        fill_value = advanced_df[column].median()
        if pd.isna(fill_value):
            fill_value = 0
        advanced_df[column] = advanced_df[column].fillna(fill_value)

In [ ]:
advanced_df.info()

In [ ]:
advanced_df.head()

In [ ]:
basic_df.head()

# Export

In [ ]:
%pip install pyarrow

basic_df.to_parquet(
    "basic_processed_data.parquet",
    engine="pyarrow",
    compression="zstd",
    index=True
)

advanced_df.to_parquet(
    "advanced_processed_data.parquet",
    engine="pyarrow",
    compression="zstd",
    index=True
)

# Classification

In [ ]:
def summarize(score, precision, recall, f1):
    print("Score", score)
    print("Precision", precision)
    print("Recall", recall)
    print("F1", f1)

## Dummy Classifier

## Decision Tree Classifier

In [ ]:
def decision_tree_classifier(X_train, X_test,y_train, y_test):
    return

## Random Forest Classifier

## Stratified K Fold Split

In [ ]:
skf = sk.model_selection.StratifiedKFold(n_splits=10, random_state=42)

data = advanced_df.copy().drop("Severity", axis = 1)
target = advanced_df.copy()["Severity"]

accuracies = []
precisions = {0:[], 1: [], 2: [], 3: []}
recalls = {0:[], 1: [], 2: [], 3: []}
f1_scores = {0:[], 1: [], 2: [], 3: []}
confusion_matrix = []


for fold, (train_index, test_index) in enumerate(skf.split(data,target), start=1):

    X_train, X_test = data.iloc[train_index], data.iloc[test_index]
    y_train, y_test = target.iloc[train_index], target.iloc[test_index]

    dummy = sk.dummy.DummyClassifier(strategy="most_frequent")
    dummy.fit(X_train,y_train)
    prediction = dummy.predict(X_test)


    dummy_confusion = sk.metrics.confusion_matrix(y_test,prediction, labels=[0,1,2,3])
    confusion_matrix.append(dummy_confusion)
    
    accuracy = dummy.score(X_test, y_test)
    accuracies.append(accuracy)

    precision, recall, f1, support = sk.metrics.precision_recall_fscore_support(
        y_test,
        prediction,
        labels=[1, 2, 3, 4],
        zero_division=0
    )

    for i,severity in enumerate([0,1,2,3]):
        precisions[severity].append(precision[i])
        recalls[severity].append(recalls[i])
        f1_scores[severity].append(f1[i])

    print("Fold: ", fold)
    print("Confusion Matrix")
    print(dummy_confusion)
    print("Accuracy", accuracy)
    for i, severity in enumerate([0,1,2,3]):
        print(f"Severity: {severity}:",
            f"Precision: {precision[i]:.3f}",
            f"Recall: {recall[i]:.3f}",
            f"F1: {f1[i]:.3f}") 

    